# Advanced Python Problems with Solutions: Class Construction, `type`, `exec`, and Metaclasses

This notebook develops the mechanics behind Python class creation through increasingly difficult exercises.

## Core ideas
- A class object is normally an instance of `type`.
- A class body executes in a namespace.
- `exec(code, globals, locals)` can populate a namespace programmatically.
- `type(name, bases, namespace)` can construct a class dynamically.
- A metaclass controls class construction.
- `__prepare__`, `__new__`, `__init__`, and `__call__` participate at different stages.
- Descriptors, decorators, inheritance, and metaclasses can interact in subtle ways.

Each problem contains:
1. a task,
2. hints or design constraints,
3. a complete solution,
4. executable checks,
5. a short explanation.

The code favors explicitness, introspection, assertions, and small reusable components.


In [1]:
import math
import inspect
from collections import OrderedDict
from types import MappingProxyType

print("Python:", __import__("sys").version.split()[0])


Python: 3.13.7


## Warm-up: Reconstruct a class manually

Before the advanced exercises, reproduce a simple class without a `class` statement.


In [2]:
class_body = '''
def __init__(self, x, y, r):
    self.x = x
    self.y = y
    self.r = r

def area(self):
    return math.pi * self.r ** 2
'''

circle_namespace = {}
exec(class_body, globals(), circle_namespace)

Circle = type("Circle", (), circle_namespace)

c = Circle(0, 0, 2)
print(type(Circle))
print(c.x, c.y, c.r)
print(c.area())

assert type(Circle) is type
assert math.isclose(c.area(), 4 * math.pi)


<class 'type'>
0 0 2
12.566370614359172


## Problem 1 — Predict the namespace produced by `exec`


Execute dynamically generated code in a separate local namespace.

Requirements:
- Create variables `base`, `scale`, and function `transform`.
- Keep them out of the module globals as much as practical.
- Inspect the resulting namespace.
- Explain why `transform.__globals__` is not the same object as the local namespace passed to `exec`.


### Hint


The third argument to `exec` is the local namespace used for assignments. Functions still retain the globals mapping supplied as the second argument.


### Solution


In [3]:
ns = {}

source = '''
base = 10
scale = 3

def transform(x):
    return (x + base) * scale
'''

# Using the same dict for globals and locals is often simpler when functions
# should later resolve dynamically-created global names from that dictionary.
exec(source, ns, ns)

print(ns.keys())
print(ns["transform"](5))

assert ns["base"] == 10
assert ns["scale"] == 3
assert ns["transform"](5) == 45
assert ns["transform"].__globals__ is ns


dict_keys(['__builtins__', 'base', 'scale', 'transform'])
45


### Why it works


A common pitfall is using two different mappings for globals and locals, then expecting a generated function to resolve local names later. For dynamically generated modules/classes, using one namespace mapping can make name resolution easier to reason about.


## Problem 2 — Build a validated class with `type`


Create a `Temperature` class dynamically using `type`.

Requirements:
- Constructor accepts Celsius.
- Reject non-numeric values.
- Provide `fahrenheit()` and `kelvin()` methods.
- Give the class a useful docstring.
- Add tests.


### Hint


Functions can be created normally and inserted into the namespace dictionary passed to `type`.


### Solution


In [4]:
def temp_init(self, celsius):
    if not isinstance(celsius, (int, float)):
        raise TypeError("celsius must be numeric")
    self.celsius = float(celsius)

def fahrenheit(self):
    return self.celsius * 9 / 5 + 32

def kelvin(self):
    return self.celsius + 273.15

Temperature = type(
    "Temperature",
    (),
    {
        "__doc__": "Temperature value stored internally in degrees Celsius.",
        "__init__": temp_init,
        "fahrenheit": fahrenheit,
        "kelvin": kelvin,
    },
)

t = Temperature(25)

assert t.celsius == 25.0
assert t.fahrenheit() == 77.0
assert math.isclose(t.kelvin(), 298.15)
assert Temperature.__doc__.startswith("Temperature")

try:
    Temperature("hot")
except TypeError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("Expected TypeError")


Expected: celsius must be numeric


### Why it works


`type(name, bases, namespace)` is enough to build a normal class object. The namespace can contain methods, attributes, descriptors, annotations, or special methods.


## Problem 3 — Generate methods in a loop without the late-binding bug


Dynamically build a class `VectorOps` with methods:

- `add_1(x)`
- `add_2(x)`
- `add_3(x)`
- `add_4(x)`
- `add_5(x)`

Each method should return `x + N`.

First understand the classic closure bug, then implement the correct version.


### Hint


Loop variables in closures are looked up later unless captured by value, for example through a default argument or a function factory.


### Solution


In [5]:
bad_namespace = {}

for n in range(1, 6):
    def adder(self, x):
        return x + n
    bad_namespace[f"add_{n}"] = adder

BadVectorOps = type("BadVectorOps", (), bad_namespace)
bad = BadVectorOps()

print([getattr(bad, f"add_{i}")(10) for i in range(1, 6)])
# All methods use the final n.

good_namespace = {}

def make_adder(n):
    def adder(self, x):
        return x + n
    return adder

for n in range(1, 6):
    good_namespace[f"add_{n}"] = make_adder(n)

VectorOps = type("VectorOps", (), good_namespace)
v = VectorOps()

results = [getattr(v, f"add_{i}")(10) for i in range(1, 6)]
print(results)

assert results == [11, 12, 13, 14, 15]


[15, 15, 15, 15, 15]
[11, 12, 13, 14, 15]


### Why it works


The important issue is closure binding time, not dynamic class creation itself. Generated methods are still ordinary function objects.


## Problem 4 — Reproduce a class statement with `exec` + `type`


Construct `BankAccount` without using a class statement for the final class.

Required API:
- `owner`
- private-ish balance `_balance`
- `deposit(amount)`
- `withdraw(amount)`
- read-only property `balance`

Validation:
- amounts must be positive
- cannot overdraw


### Hint


You can define functions in a source string, execute them into a namespace, add a `property`, then pass the namespace to `type`.


### Solution


In [6]:
account_ns = {}

account_body = '''
def __init__(self, owner, opening_balance=0):
    if opening_balance < 0:
        raise ValueError("opening_balance cannot be negative")
    self.owner = owner
    self._balance = float(opening_balance)

def deposit(self, amount):
    if amount <= 0:
        raise ValueError("amount must be positive")
    self._balance += amount
    return self._balance

def withdraw(self, amount):
    if amount <= 0:
        raise ValueError("amount must be positive")
    if amount > self._balance:
        raise ValueError("insufficient funds")
    self._balance -= amount
    return self._balance

def _get_balance(self):
    return self._balance
'''

exec(account_body, globals(), account_ns)
account_ns["balance"] = property(account_ns.pop("_get_balance"))

BankAccount = type("BankAccount", (), account_ns)

acc = BankAccount("Ada", 100)
assert acc.deposit(25) == 125
assert acc.withdraw(40) == 85
assert acc.balance == 85

try:
    acc.balance = 999
except AttributeError:
    pass
else:
    raise AssertionError("balance should be read-only")


### Why it works


The generated class behaves like a class written with normal syntax because the end product is still a class object with the same kinds of namespace entries.


## Problem 5 — Use inheritance in dynamic class construction


Create:
- a normal base class `Shape`
- a dynamically-created subclass `Rectangle`

`Rectangle` must:
- inherit from `Shape`
- implement `area`
- implement `perimeter`
- preserve `isinstance(rect, Shape)`


### Hint


Pass a tuple of base classes as the second argument to `type`.


### Solution


In [7]:
class Shape:
    def area(self):
        raise NotImplementedError

def rectangle_init(self, width, height):
    if width <= 0 or height <= 0:
        raise ValueError("dimensions must be positive")
    self.width = width
    self.height = height

def rectangle_area(self):
    return self.width * self.height

def rectangle_perimeter(self):
    return 2 * (self.width + self.height)

Rectangle = type(
    "Rectangle",
    (Shape,),
    {
        "__init__": rectangle_init,
        "area": rectangle_area,
        "perimeter": rectangle_perimeter,
    },
)

r = Rectangle(3, 4)

assert isinstance(r, Shape)
assert r.area() == 12
assert r.perimeter() == 14
assert Rectangle.__mro__[:2] == (Rectangle, Shape)

print(Rectangle.__mro__)


(<class '__main__.Rectangle'>, <class '__main__.Shape'>, <class 'object'>)


### Why it works


Dynamic creation does not bypass normal inheritance. Python computes an MRO for the generated class exactly as it does for a class statement.


## Problem 6 — Multiple inheritance and MRO reasoning


Build `FlyingSwimmingDuck` dynamically from `Flyer` and `Swimmer`.

Each base defines `move`. Use cooperative `super()` so the final output reflects MRO traversal.

Then inspect `__mro__`.


### Hint


For cooperative multiple inheritance, every participating implementation should call `super()` consistently.


### Solution


In [8]:
class Root:
    def move(self):
        return ["Root"]

class Flyer(Root):
    def move(self):
        return ["Flyer"] + super().move()

class Swimmer(Root):
    def move(self):
        return ["Swimmer"] + super().move()

FlyingSwimmingDuck = type(
    "FlyingSwimmingDuck",
    (Flyer, Swimmer),
    {}
)

duck = FlyingSwimmingDuck()

print(FlyingSwimmingDuck.__mro__)
print(duck.move())

assert duck.move() == ["Flyer", "Swimmer", "Root"]


(<class '__main__.FlyingSwimmingDuck'>, <class '__main__.Flyer'>, <class '__main__.Swimmer'>, <class '__main__.Root'>, <class 'object'>)
['Flyer', 'Swimmer', 'Root']


### Why it works


The generated class uses C3 linearization just like any other Python class. The order of bases matters.


## Problem 7 — Write a metaclass that injects methods


Create a metaclass `ReprMeta`.

For every class using it:
- if the class does not define `__repr__`,
- inject a representation that displays public instance attributes.

Do not overwrite an explicitly supplied `__repr__`.


### Hint


A metaclass can mutate the namespace before delegating to `type.__new__`.


### Solution


In [9]:
class ReprMeta(type):
    def __new__(mcls, name, bases, namespace):
        if "__repr__" not in namespace:
            def __repr__(self):
                public = {
                    k: v for k, v in vars(self).items()
                    if not k.startswith("_")
                }
                args = ", ".join(f"{k}={v!r}" for k, v in public.items())
                return f"{type(self).__name__}({args})"

            namespace["__repr__"] = __repr__

        return super().__new__(mcls, name, bases, namespace)

class Person(metaclass=ReprMeta):
    def __init__(self, name, age):
        self.name = name
        self.age = age
        self._token = "hidden"

p = Person("Grace", 37)
print(repr(p))
assert repr(p) == "Person(name='Grace', age=37)"


Person(name='Grace', age=37)


### Why it works


Metaclasses can enforce or supply class-level conventions at class creation time.


## Problem 8 — Validate class definitions in a metaclass


Create an `InterfaceMeta` metaclass requiring every concrete subclass to define a callable `run`.

Rules:
- the base class itself is exempt,
- subclasses missing `run` fail immediately at class creation time,
- a non-callable `run` also fails.


### Hint


Validation belongs naturally in `__new__` or `__init__` of the metaclass because the class namespace is available there.


### Solution


In [10]:
class InterfaceMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        is_root = not bases or all(base is object for base in bases)
        if not is_root:
            candidate = namespace.get("run")
            if candidate is None or not callable(candidate):
                raise TypeError(f"{name} must define a callable run(self, ...) method")

        return cls

class TaskBase(metaclass=InterfaceMeta):
    pass

class GoodTask(TaskBase):
    def run(self):
        return "done"

assert GoodTask().run() == "done"

try:
    class BadTask(TaskBase):
        run = 123
except TypeError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("BadTask should not have been created")


Expected: BadTask must define a callable run(self, ...) method


### Why it works


Failing during class creation catches structural problems before instances ever exist.


## Problem 9 — Track declaration order with `__prepare__`


Create a metaclass that records the order in which public attributes are declared.

Store the result on the class as `__definition_order_public__`.

Ignore names starting with `_`.


### Hint


`__prepare__` supplies the mapping used as the class namespace before the class body executes.


### Solution


In [11]:
class OrderedMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases):
        # Modern dict preserves insertion order, but OrderedDict makes intent explicit.
        return OrderedDict()

    def __new__(mcls, name, bases, namespace):
        public_names = [
            key for key in namespace
            if not key.startswith("_")
        ]
        namespace["__definition_order_public__"] = tuple(public_names)
        return super().__new__(mcls, name, bases, dict(namespace))

class Config(metaclass=OrderedMeta):
    host = "localhost"
    port = 8080

    def connect(self):
        return f"{self.host}:{self.port}"

print(Config.__definition_order_public__)
assert Config.__definition_order_public__ == ("host", "port", "connect")


('host', 'port', 'connect')


### Why it works


`__prepare__` is useful when the class body needs a specialized namespace: ordered mappings, duplicate-detection mappings, logging mappings, DSL-like mappings, and more.


## Problem 10 — Detect duplicate names during class definition


Implement a custom namespace mapping that raises an error if the same user-defined name is assigned twice in one class body.

Allow interpreter-managed names such as `__module__` and `__qualname__`.


### Hint


Return the custom mapping from `__prepare__`.


### Solution


In [12]:
class NoDuplicateDict(dict):
    allowed_reassignments = {"__module__", "__qualname__", "__annotations__"}

    def __setitem__(self, key, value):
        if key in self and key not in self.allowed_reassignments:
            raise TypeError(f"duplicate class attribute: {key}")
        super().__setitem__(key, value)

class NoDuplicateMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases):
        return NoDuplicateDict()

class Valid(metaclass=NoDuplicateMeta):
    x = 1
    y = 2

assert Valid.x == 1

try:
    class Invalid(metaclass=NoDuplicateMeta):
        x = 1
        x = 2
except TypeError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("duplicate should have been rejected")


Expected: duplicate class attribute: x


### Why it works


This demonstrates that a metaclass can influence execution of the class body even before the final class object exists.


## Problem 11 — Build a descriptor registry automatically


Create a descriptor `Field` and a metaclass `ModelMeta`.

Each `Field`:
- knows its attribute name via `__set_name__`,
- validates values with a supplied validator.

The metaclass must:
- gather fields declared on the class,
- include inherited fields,
- store them in `__fields__`.

Then create a `User` model.


### Hint


Descriptors are placed in the class namespace first; `__set_name__` is called during class creation.


### Solution


In [13]:
class Field:
    def __init__(self, validator):
        self.validator = validator
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not self.validator(value):
            raise ValueError(f"invalid value for {self.name}: {value!r}")
        instance.__dict__[self.name] = value

class ModelMeta(type):
    def __new__(mcls, name, bases, namespace):
        inherited = {}
        for base in bases:
            inherited.update(getattr(base, "__fields__", {}))

        declared = {
            key: value
            for key, value in namespace.items()
            if isinstance(value, Field)
        }

        cls = super().__new__(mcls, name, bases, namespace)
        cls.__fields__ = {**inherited, **declared}
        return cls

class Model(metaclass=ModelMeta):
    pass

class User(Model):
    name = Field(lambda v: isinstance(v, str) and bool(v.strip()))
    age = Field(lambda v: isinstance(v, int) and 0 <= v <= 150)

    def __init__(self, name, age):
        self.name = name
        self.age = age

u = User("Linus", 30)
print(User.__fields__)
assert set(User.__fields__) == {"name", "age"}

try:
    u.age = -1
except ValueError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("invalid age should fail")


{'name': <__main__.Field object at 0x0000029E311030E0>, 'age': <__main__.Field object at 0x0000029E3119C190>}
Expected: invalid value for age: -1


### Why it works


This pattern is common in ORMs, serializers, validation libraries, command frameworks, and configuration systems.


## Problem 12 — Override metaclass `__call__` to implement instance caching


Implement a metaclass `CachedMeta` so repeated construction with the same positional/keyword arguments returns the same instance.

Use it for an immutable-ish `Symbol` class.

Caveat:
- the cache key must be hashable.


### Hint


Calling a class invokes the metaclass's `__call__`, which normally coordinates instance creation through `__new__` and `__init__`.


### Solution


In [14]:
class CachedMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        cls.__instance_cache__ = {}
        return cls

    def __call__(cls, *args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cls.__instance_cache__:
            cls.__instance_cache__[key] = super().__call__(*args, **kwargs)
        return cls.__instance_cache__[key]

class Symbol(metaclass=CachedMeta):
    def __init__(self, name, category="generic"):
        self.name = name
        self.category = category

a = Symbol("x")
b = Symbol("x")
c = Symbol("x", category="math")

assert a is b
assert a is not c

print(id(a), id(b), id(c))


2878451233664 2878451233664 2878451860496


### Why it works


Metaclass `__call__` is about creating instances of classes governed by that metaclass, while metaclass `__new__` is about creating the class objects themselves.


## Problem 13 — Implement a tiny class factory


Write `record_type(name, fields)` returning a new class.

Generated class requirements:
- positional constructor matching the field order,
- readable `__repr__`,
- structural `__eq__`,
- reject the wrong number of arguments,
- expose `_fields`.


### Hint


Generate closures over the validated field tuple, then build the result using `type`.


### Solution


In [15]:
def record_type(name, fields):
    fields = tuple(fields)

    if len(set(fields)) != len(fields):
        raise ValueError("field names must be unique")

    if not all(isinstance(field, str) and field.isidentifier() for field in fields):
        raise ValueError("all fields must be valid identifiers")

    def __init__(self, *values):
        if len(values) != len(fields):
            raise TypeError(
                f"{name} expected {len(fields)} arguments, got {len(values)}"
            )
        for field, value in zip(fields, values):
            setattr(self, field, value)

    def __repr__(self):
        pairs = ", ".join(f"{f}={getattr(self, f)!r}" for f in fields)
        return f"{name}({pairs})"

    def __eq__(self, other):
        if type(other) is not type(self):
            return NotImplemented
        return all(getattr(self, f) == getattr(other, f) for f in fields)

    return type(
        name,
        (),
        {
            "_fields": fields,
            "__init__": __init__,
            "__repr__": __repr__,
            "__eq__": __eq__,
        },
    )

Point = record_type("Point", ["x", "y"])
p1 = Point(2, 3)
p2 = Point(2, 3)

print(p1)
assert p1 == p2
assert repr(p1) == "Point(x=2, y=3)"
assert Point._fields == ("x", "y")


Point(x=2, y=3)


### Why it works


This is a small-scale example of metaprogramming: code creates classes tailored to input data.


## Problem 14 — Compare class decorators and metaclasses


Create:
1. a class decorator `tagged(label)`,
2. a metaclass `AuditMeta`.

The decorator adds `tag`.
The metaclass adds `created_by_meta = True`.

Apply both to one class and inspect the result.

Then explain the conceptual timing difference.


### Hint


A metaclass participates in construction of the class object. A class decorator receives the already-created class object and returns a class object.


### Solution


In [16]:
def tagged(label):
    def decorate(cls):
        cls.tag = label
        return cls
    return decorate

class AuditMeta(type):
    def __new__(mcls, name, bases, namespace):
        namespace["created_by_meta"] = True
        return super().__new__(mcls, name, bases, namespace)

@tagged("service")
class Service(metaclass=AuditMeta):
    pass

assert Service.tag == "service"
assert Service.created_by_meta is True

print(Service.tag, Service.created_by_meta)


service True


### Why it works


Use a class decorator when post-processing the finished class is sufficient. Use a metaclass when you need deeper control over namespace preparation, validation, construction, or instance creation.


## Problem 15 — Advanced metaclass conflict resolution


Given two unrelated metaclasses `MetaA` and `MetaB`, classes using them cannot normally be multiply inherited without resolving the metaclass relationship.

Task:
- demonstrate the conflict,
- create a combined metaclass,
- build a subclass inheriting from both base classes.


### Hint


The selected metaclass must be a subclass of the metaclasses of all base classes.


### Solution


In [17]:
class MetaA(type):
    pass

class MetaB(type):
    pass

class A(metaclass=MetaA):
    pass

class B(metaclass=MetaB):
    pass

try:
    class Broken(A, B):
        pass
except TypeError as exc:
    print("Expected conflict:", exc)
else:
    raise AssertionError("Expected metaclass conflict")

class MetaAB(MetaA, MetaB):
    pass

class Combined(A, B, metaclass=MetaAB):
    pass

assert isinstance(Combined, MetaA)
assert isinstance(Combined, MetaB)
print(type(Combined), Combined.__mro__)


Expected conflict: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases
<class '__main__.MetaAB'> (<class '__main__.Combined'>, <class '__main__.A'>, <class '__main__.B'>, <class 'object'>)


### Why it works


Metaclass conflicts are a consequence of Python needing one metaclass that is compatible with every base class's metaclass.


## Problem 16 — Dynamic plugin registration via metaclass


Build an automatic registry.

Requirements:
- abstract-ish root `Plugin` is not registered,
- each concrete subclass declares a unique `plugin_name`,
- duplicate names raise an error,
- provide `Plugin.create(name, **kwargs)`.


### Hint


The registry can live on the metaclass so it is shared across the class hierarchy.


### Solution


In [18]:
class PluginMeta(type):
    registry = {}

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        if name != "Plugin":
            plugin_name = namespace.get("plugin_name")
            if not isinstance(plugin_name, str) or not plugin_name:
                raise TypeError(f"{name} must define non-empty plugin_name")

            if plugin_name in mcls.registry:
                raise ValueError(f"duplicate plugin_name: {plugin_name}")

            mcls.registry[plugin_name] = cls

        return cls

class Plugin(metaclass=PluginMeta):
    @classmethod
    def create(cls, name, **kwargs):
        try:
            plugin_cls = PluginMeta.registry[name]
        except KeyError:
            raise KeyError(f"unknown plugin: {name}")
        return plugin_cls(**kwargs)

class UpperPlugin(Plugin):
    plugin_name = "upper"

    def __init__(self, prefix=""):
        self.prefix = prefix

    def run(self, text):
        return self.prefix + text.upper()

class RepeatPlugin(Plugin):
    plugin_name = "repeat"

    def __init__(self, times=2):
        self.times = times

    def run(self, text):
        return text * self.times

plugin = Plugin.create("upper", prefix="> ")
assert plugin.run("hello") == "> HELLO"
print(PluginMeta.registry)


{'upper': <class '__main__.UpperPlugin'>, 'repeat': <class '__main__.RepeatPlugin'>}


### Why it works


Registration at class creation time eliminates a separate manual registration call and can enforce uniqueness early.


## Problem 17 — Metaclass-generated `__slots__`


Create a metaclass that reads a declaration:

    fields = ("x", "y")

and converts it into `__slots__` before class creation.

The final class should:
- use slots for the listed names,
- not have a normal instance `__dict__`,
- expose `fields` as a tuple.


### Hint


`__slots__` must be present while the class itself is being created. Adding it later is too late.


### Solution


In [19]:
class SlotsMeta(type):
    def __new__(mcls, name, bases, namespace):
        declared = tuple(namespace.get("fields", ()))
        if declared:
            namespace["fields"] = declared
            namespace["__slots__"] = declared
        return super().__new__(mcls, name, bases, namespace)

class SlotPoint(metaclass=SlotsMeta):
    fields = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

sp = SlotPoint(1, 2)

assert (sp.x, sp.y) == (1, 2)
assert not hasattr(sp, "__dict__")

try:
    sp.z = 3
except AttributeError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("slot class should reject arbitrary attributes")


Expected: 'SlotPoint' object has no attribute 'z' and no __dict__ for setting new attributes


### Why it works


This is a case where the timing of metaclass intervention matters: the namespace must be changed before delegating to `type.__new__`.


## Problem 18 — Build a mini declarative schema system


Combine descriptors and metaclasses into a tiny schema framework.

Support field types:
- `StringField(min_length=...)`
- `IntegerField(min_value=..., max_value=...)`

Requirements:
- collect fields automatically,
- constructor accepts keyword arguments,
- reject unknown keys,
- reject missing keys,
- validate all assignments,
- provide `to_dict()`.


### Hint


Separate concerns:
- descriptors validate individual values,
- the metaclass collects schema metadata,
- the base model implements generic construction/serialization.


### Solution


In [20]:
class BaseField:
    def __init__(self):
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def validate(self, value):
        raise NotImplementedError

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        self.validate(value)
        instance.__dict__[self.name] = value

class StringField(BaseField):
    def __init__(self, min_length=0):
        super().__init__()
        self.min_length = min_length

    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be str")
        if len(value) < self.min_length:
            raise ValueError(
                f"{self.name} must have length >= {self.min_length}"
            )

class IntegerField(BaseField):
    def __init__(self, min_value=None, max_value=None):
        super().__init__()
        self.min_value = min_value
        self.max_value = max_value

    def validate(self, value):
        if not isinstance(value, int) or isinstance(value, bool):
            raise TypeError(f"{self.name} must be int")
        if self.min_value is not None and value < self.min_value:
            raise ValueError(f"{self.name} must be >= {self.min_value}")
        if self.max_value is not None and value > self.max_value:
            raise ValueError(f"{self.name} must be <= {self.max_value}")

class SchemaMeta(type):
    def __new__(mcls, name, bases, namespace):
        fields = {}

        for base in bases:
            fields.update(getattr(base, "__fields__", {}))

        fields.update({
            key: value
            for key, value in namespace.items()
            if isinstance(value, BaseField)
        })

        cls = super().__new__(mcls, name, bases, namespace)
        cls.__fields__ = fields
        return cls

class Schema(metaclass=SchemaMeta):
    def __init__(self, **kwargs):
        expected = set(self.__fields__)
        supplied = set(kwargs)

        missing = expected - supplied
        unknown = supplied - expected

        if missing:
            raise TypeError(f"missing fields: {sorted(missing)}")
        if unknown:
            raise TypeError(f"unknown fields: {sorted(unknown)}")

        for name, value in kwargs.items():
            setattr(self, name, value)

    def to_dict(self):
        return {name: getattr(self, name) for name in self.__fields__}

class Product(Schema):
    name = StringField(min_length=2)
    price = IntegerField(min_value=0)
    stock = IntegerField(min_value=0, max_value=1_000_000)

product = Product(name="Keyboard", price=120, stock=15)

assert product.to_dict() == {
    "name": "Keyboard",
    "price": 120,
    "stock": 15,
}

print(product.to_dict())

for bad_kwargs in [
    {"name": "K", "price": 120, "stock": 15},
    {"name": "Keyboard", "price": -1, "stock": 15},
    {"name": "Keyboard", "price": 120},
    {"name": "Keyboard", "price": 120, "stock": 15, "color": "black"},
]:
    try:
        Product(**bad_kwargs)
    except (TypeError, ValueError) as exc:
        print("Expected:", exc)
    else:
        raise AssertionError(f"Expected failure for {bad_kwargs}")


{'name': 'Keyboard', 'price': 120, 'stock': 15}
Expected: name must have length >= 2
Expected: price must be >= 0
Expected: missing fields: ['stock']
Expected: unknown fields: ['color']


### Why it works


This exercise combines several class-construction mechanisms in a realistic pattern while keeping each responsibility isolated.


# Challenge Set — Try Before Reading the Solutions

The following compact exercises are intentionally less scaffolded.


## Challenge A — Dynamic immutable record

Write a function `frozen_record(name, fields)` that creates a class whose fields:
- are provided positionally,
- can be read,
- cannot be reassigned after initialization,
- have a deterministic `repr`.

Try to solve it before opening the solution cell.


In [21]:
def frozen_record(name, fields):
    fields = tuple(fields)

    def __init__(self, *values):
        if len(values) != len(fields):
            raise TypeError(f"expected {len(fields)} values")
        object.__setattr__(self, "_locked", False)
        for field, value in zip(fields, values):
            object.__setattr__(self, field, value)
        object.__setattr__(self, "_locked", True)

    def __setattr__(self, key, value):
        if getattr(self, "_locked", False):
            raise AttributeError(f"{name} is immutable")
        object.__setattr__(self, key, value)

    def __repr__(self):
        args = ", ".join(f"{f}={getattr(self, f)!r}" for f in fields)
        return f"{name}({args})"

    return type(
        name,
        (),
        {
            "_fields": fields,
            "__init__": __init__,
            "__setattr__": __setattr__,
            "__repr__": __repr__,
        },
    )

FrozenPoint = frozen_record("FrozenPoint", ("x", "y"))
fp = FrozenPoint(10, 20)
assert repr(fp) == "FrozenPoint(x=10, y=20)"

try:
    fp.x = 99
except AttributeError:
    pass
else:
    raise AssertionError("object should be immutable")


## Challenge B — Automatic method wrapping

Write a metaclass that wraps every public method defined directly in a class so that each call increments a class-level counter.

The solution below keeps a per-method count.


In [22]:
class CountingMeta(type):
    def __new__(mcls, name, bases, namespace):
        counts = {}

        def wrap(method_name, fn):
            counts[method_name] = 0

            def wrapped(self, *args, **kwargs):
                counts[method_name] += 1
                return fn(self, *args, **kwargs)

            wrapped.__name__ = fn.__name__
            wrapped.__doc__ = fn.__doc__
            return wrapped

        transformed = dict(namespace)

        for key, value in namespace.items():
            if key.startswith("_"):
                continue
            if inspect.isfunction(value):
                transformed[key] = wrap(key, value)

        transformed["__call_counts__"] = counts
        return super().__new__(mcls, name, bases, transformed)

class Calculator(metaclass=CountingMeta):
    def add(self, a, b):
        return a + b

    def mul(self, a, b):
        return a * b

calc = Calculator()

assert calc.add(1, 2) == 3
assert calc.add(3, 4) == 7
assert calc.mul(5, 6) == 30

print(Calculator.__call_counts__)
assert Calculator.__call_counts__ == {"add": 2, "mul": 1}


{'add': 2, 'mul': 1}


## Challenge C — Metaclass + inheritance field override

Extend the schema approach so a subclass can replace an inherited field definition while preserving all other inherited fields.


In [23]:
class PositiveInt(IntegerField):
    def __init__(self):
        super().__init__(min_value=1)

class InventoryItem(Product):
    # Override inherited stock validation:
    stock = PositiveInt()

item = InventoryItem(name="Mouse", price=25, stock=5)

assert isinstance(InventoryItem.__fields__["stock"], PositiveInt)
assert set(InventoryItem.__fields__) == {"name", "price", "stock"}

try:
    InventoryItem(name="Mouse", price=25, stock=0)
except ValueError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("stock=0 should fail for InventoryItem")


Expected: stock must be >= 1


# Deep-Dive Introspection

Use these cells to inspect what Python actually created.


In [24]:
targets = [Circle, Temperature, Rectangle, Person, User, Product]

for cls in targets:
    print("=" * 70)
    print("CLASS:", cls.__name__)
    print("METACLASS:", type(cls).__name__)
    print("BASES:", cls.__bases__)
    print("MRO:", cls.__mro__)
    print("NAMESPACE KEYS:", sorted(cls.__dict__.keys()))


CLASS: Circle
METACLASS: type
BASES: (<class 'object'>,)
MRO: (<class '__main__.Circle'>, <class 'object'>)
NAMESPACE KEYS: ['__dict__', '__doc__', '__init__', '__module__', '__weakref__', 'area']
CLASS: Temperature
METACLASS: type
BASES: (<class 'object'>,)
MRO: (<class '__main__.Temperature'>, <class 'object'>)
NAMESPACE KEYS: ['__dict__', '__doc__', '__init__', '__module__', '__weakref__', 'fahrenheit', 'kelvin']
CLASS: Rectangle
METACLASS: type
BASES: (<class '__main__.Shape'>,)
MRO: (<class '__main__.Rectangle'>, <class '__main__.Shape'>, <class 'object'>)
NAMESPACE KEYS: ['__doc__', '__init__', '__module__', 'area', 'perimeter']
CLASS: Person
METACLASS: ReprMeta
BASES: (<class 'object'>,)
MRO: (<class '__main__.Person'>, <class 'object'>)
NAMESPACE KEYS: ['__dict__', '__doc__', '__firstlineno__', '__init__', '__module__', '__repr__', '__static_attributes__', '__weakref__']
CLASS: User
METACLASS: ModelMeta
BASES: (<class '__main__.Model'>,)
MRO: (<class '__main__.User'>, <class '_

In [25]:
# Class dictionaries are exposed as read-only mapping proxies.
print(type(Product.__dict__))
assert isinstance(Product.__dict__, MappingProxyType)

try:
    Product.__dict__["new_name"] = 123
except TypeError as exc:
    print("Expected:", exc)


<class 'mappingproxy'>
Expected: 'mappingproxy' object does not support item assignment


In [26]:
# But setattr on the class is allowed unless a metaclass prevents it.
Product.category = "hardware"
assert Product.category == "hardware"
print(Product.__dict__["category"])


hardware


# Best-Practice Notes

- Prefer normal `class` syntax for ordinary classes.
- Use `type(...)` when class structure truly needs to be generated from data.
- Use class decorators for lightweight post-processing.
- Use metaclasses when you need control over class namespace preparation, validation, construction, or class-call behavior.
- Keep metaclasses small; they can make control flow harder to follow.
- Separate validation, registration, descriptors, and instance behavior when possible.
- Prefer `super()` consistently in cooperative multiple inheritance.
- Be careful with closures created inside loops.
- Treat `exec` as powerful and dangerous: never execute untrusted strings.
- Add assertions/tests around metaprogramming because failures can occur at import/class-definition time.
- Preserve function metadata with `functools.wraps` in production wrappers.


# Final Master Problem

Design a small event system with all of the following:

1. An `EventField` descriptor validates one attribute.
2. An `EventMeta` metaclass:
   - collects fields,
   - requires a string `event_type`,
   - registers subclasses by `event_type`,
   - rejects duplicate event types.
3. `Event.from_dict(data)`:
   - reads the key `"type"`,
   - selects the right event subclass,
   - validates payload fields,
   - returns an instance.
4. Add at least two concrete event types.
5. Add tests for valid events, unknown event type, missing field, and invalid field value.

A reference solution follows.


In [27]:
class EventField:
    def __init__(self, validator):
        self.validator = validator
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not self.validator(value):
            raise ValueError(f"invalid {self.name}: {value!r}")
        instance.__dict__[self.name] = value

class EventMeta(type):
    registry = {}

    def __new__(mcls, name, bases, namespace):
        inherited = {}
        for base in bases:
            inherited.update(getattr(base, "__fields__", {}))

        declared = {
            key: value
            for key, value in namespace.items()
            if isinstance(value, EventField)
        }

        cls = super().__new__(mcls, name, bases, namespace)
        cls.__fields__ = {**inherited, **declared}

        if name != "Event":
            event_type = namespace.get("event_type")

            if not isinstance(event_type, str) or not event_type:
                raise TypeError(f"{name} must define event_type")

            if event_type in mcls.registry:
                raise ValueError(f"duplicate event_type: {event_type}")

            mcls.registry[event_type] = cls

        return cls

class Event(metaclass=EventMeta):
    def __init__(self, **payload):
        expected = set(self.__fields__)
        supplied = set(payload)

        missing = expected - supplied
        extra = supplied - expected

        if missing:
            raise TypeError(f"missing fields: {sorted(missing)}")
        if extra:
            raise TypeError(f"unknown fields: {sorted(extra)}")

        for name, value in payload.items():
            setattr(self, name, value)

    @classmethod
    def from_dict(cls, data):
        if not isinstance(data, dict):
            raise TypeError("data must be a dict")

        event_type = data.get("type")
        if event_type not in EventMeta.registry:
            raise KeyError(f"unknown event type: {event_type!r}")

        event_cls = EventMeta.registry[event_type]
        payload = {k: v for k, v in data.items() if k != "type"}
        return event_cls(**payload)

    def to_dict(self):
        return {
            "type": self.event_type,
            **{name: getattr(self, name) for name in self.__fields__},
        }

class UserCreated(Event):
    event_type = "user.created"

    user_id = EventField(lambda v: isinstance(v, int) and v > 0)
    username = EventField(lambda v: isinstance(v, str) and len(v) >= 3)

class OrderPaid(Event):
    event_type = "order.paid"

    order_id = EventField(lambda v: isinstance(v, int) and v > 0)
    amount = EventField(
        lambda v: isinstance(v, (int, float))
        and not isinstance(v, bool)
        and v > 0
    )

evt1 = Event.from_dict({
    "type": "user.created",
    "user_id": 7,
    "username": "ada",
})

evt2 = Event.from_dict({
    "type": "order.paid",
    "order_id": 101,
    "amount": 49.99,
})

assert isinstance(evt1, UserCreated)
assert isinstance(evt2, OrderPaid)
assert evt1.to_dict()["username"] == "ada"
assert evt2.to_dict()["amount"] == 49.99

bad_cases = [
    {"type": "unknown.event", "x": 1},
    {"type": "user.created", "user_id": 7},
    {"type": "user.created", "user_id": -1, "username": "ada"},
]

for case in bad_cases:
    try:
        Event.from_dict(case)
    except (KeyError, TypeError, ValueError) as exc:
        print("Expected:", type(exc).__name__, exc)
    else:
        raise AssertionError(f"Expected failure for {case}")

print("Registered events:", sorted(EventMeta.registry))


Expected: KeyError "unknown event type: 'unknown.event'"
Expected: TypeError missing fields: ['username']
Expected: ValueError invalid user_id: -1
Registered events: ['order.paid', 'user.created']


# End-of-Notebook Checklist

After completing the notebook, you should be able to explain:

- what executes when Python sees a class body,
- why the class namespace is central,
- what `type(name, bases, namespace)` does,
- how generated methods behave like normal functions,
- how inheritance and MRO still apply to dynamic classes,
- when `__prepare__` runs,
- how a metaclass can validate or modify a class,
- the distinction between metaclass `__new__` and `__call__`,
- how descriptors interact with class creation,
- when to choose class decorators versus metaclasses,
- why metaclass conflicts occur,
- how real frameworks use these ideas for registries, schemas, models, and DSLs.
